In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType


In [0]:
spark.conf.set("spark.sql.session.timeZone", "Asia/Seoul")

In [0]:
# =========================================================
# 1. BRZ MONTH temp 로드
# =========================================================
raw_df = spark.table(
    "hive_metastore.demo_airstatus_bronze.BRZ_temp_seoul_air_quality_month"
)

In [0]:
# =========================================================
# 2. dataTime 문자열 정제
#    - "-" 제거
#    - "24:00" → 다음날 "00:00" (타입 변환 전!)
# =========================================================
clean_df = (
    raw_df
    .filter(col("dataTime") != "-")

    .withColumn(
        "dataTime_fixed",
        when(
            col("dataTime").endswith(" 24:00"),
            to_utc_timestamp(
                concat(
                    date_add(to_date(substring(col("dataTime"), 1, 10)), 1),
                    lit(" 00:00")
                ),
                "Asia/Seoul"
            )
        ).otherwise(
            to_utc_timestamp(col("dataTime"), "Asia/Seoul")
        )
    )
)

In [0]:
display(clean_df[clean_df['stationName']=="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value,dataTime_fixed
중구,2026-05-04 13:00,73,2,25,14,2026-05-04T04:00:00+09:00
중구,2026-05-04 12:00,71,2,21,11,2026-05-04T03:00:00+09:00
중구,2026-05-04 11:00,66,2,17,8,2026-05-04T02:00:00+09:00
중구,2026-05-04 10:00,60,2,12,4,2026-05-04T01:00:00+09:00
중구,2026-05-04 09:00,57,2,9,6,2026-05-04T00:00:00+09:00
중구,2026-05-04 08:00,54,2,9,5,2026-05-03T23:00:00+09:00
중구,2026-05-04 07:00,53,2,10,3,2026-05-03T22:00:00+09:00
중구,2026-05-04 06:00,56,2,10,6,2026-05-03T21:00:00+09:00
중구,2026-05-04 05:00,60,2,9,4,2026-05-03T20:00:00+09:00
중구,2026-05-04 04:00,61,2,9,7,2026-05-03T19:00:00+09:00


In [0]:
# =========================================================
# 3. 값 컬럼 정제 (타입 변환)
# =========================================================
clean_df = (
    clean_df
    .withColumn(
        "khaiValue",
        coalesce(regexp_replace(col("khaiValue"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "khaiGrade",
        coalesce(regexp_replace(col("khaiGrade"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "pm10Value",
        coalesce(regexp_replace(col("pm10Value"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "pm25Value",
        coalesce(regexp_replace(col("pm25Value"), "-", "").cast(IntegerType()), lit(0))
    )
)

In [0]:
display(clean_df[clean_df['stationName']=="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value,dataTime_fixed
중구,2026-05-04 13:00,73,2,25,14,2026-05-04T04:00:00+09:00
중구,2026-05-04 12:00,71,2,21,11,2026-05-04T03:00:00+09:00
중구,2026-05-04 11:00,66,2,17,8,2026-05-04T02:00:00+09:00
중구,2026-05-04 10:00,60,2,12,4,2026-05-04T01:00:00+09:00
중구,2026-05-04 09:00,57,2,9,6,2026-05-04T00:00:00+09:00
중구,2026-05-04 08:00,54,2,9,5,2026-05-03T23:00:00+09:00
중구,2026-05-04 07:00,53,2,10,3,2026-05-03T22:00:00+09:00
중구,2026-05-04 06:00,56,2,10,6,2026-05-03T21:00:00+09:00
중구,2026-05-04 05:00,60,2,9,4,2026-05-03T20:00:00+09:00
중구,2026-05-04 04:00,61,2,9,7,2026-05-03T19:00:00+09:00


In [0]:
# =========================================================
# 4. dataTime 컬럼 확정 + NULL 제거
# =========================================================
final_df = (
    clean_df
    .select(
        "stationName",
        col("dataTime_fixed").alias("dataTime"),
        "khaiValue",
        "khaiGrade",
        "pm10Value",
        "pm25Value"
    )
    .filter(col("dataTime").isNotNull())
)

In [0]:
# =========================================================
# 5. 중복 제거 (BACKFILL 핵심)
# =========================================================
final_df = final_df.dropDuplicates(["stationName", "dataTime"])

In [0]:
# =========================================================
# 6. 파생 컬럼 생성
# =========================================================
fact_df = (
    final_df
    .withColumn("year", year("dataTime"))
    .withColumn("month", month("dataTime"))
    .withColumn("day", dayofmonth("dataTime"))
    .withColumn("hour", hour("dataTime"))
)

In [0]:
display(fact_df[fact_df['stationName']=="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value,year,month,day,hour
중구,2026-05-04T01:00:00+09:00,60,2,12,4,2026,5,4,1
중구,2026-05-03T17:00:00+09:00,55,2,4,4,2026,5,3,17
중구,2026-05-01T07:00:00+09:00,86,2,38,13,2026,5,1,7
중구,2026-05-02T11:00:00+09:00,132,3,58,37,2026,5,2,11
중구,2026-05-01T03:00:00+09:00,79,2,38,16,2026,5,1,3
중구,2026-04-30T14:00:00+09:00,70,2,21,10,2026,4,30,14
중구,2026-05-01T12:00:00+09:00,78,2,48,29,2026,5,1,12
중구,2026-05-04T02:00:00+09:00,66,2,17,8,2026,5,4,2
중구,2026-05-02T06:00:00+09:00,143,3,73,49,2026,5,2,6
중구,2026-05-01T00:00:00+09:00,67,2,26,13,2026,5,1,0


In [0]:
# =========================================================
# 7. SLV MONTH temp overwrite
# =========================================================
fact_df.write.mode("overwrite").saveAsTable(
  "hive_metastore.demo_airstatus_silver.SLV_temp_fact_air_quality_month"
)



In [0]:

# =========================================================
# 8. SLV MAIN append (최초 반영 지점)
# =========================================================
fact_df.write.mode("append").saveAsTable(
  "hive_metastore.demo_airstatus_silver.SLV_fact_air_quality"
)